# 🦴 Auxetic Bone Implant — Parameter Sweep Pipeline

This notebook sets up and runs the full auxetic plate pipeline on Google Colab.

**Pipeline stages:** Case Generation → Geometry (CadQuery) → Meshing (gmsh) → Solver Export (CalculiX) → Ranking & Reporting

---

## ⚠️ Important: Run cells in order. Cell 1 will restart the kernel automatically.

## Step 1: Install Conda (kernel will restart automatically)

In [ ]:
# Install condacolab to get access to conda-forge packages (CadQuery, gmsh, CalculiX)
# NOTE: This cell WILL restart the kernel. That is expected.
!pip install -q condacolab
import condacolab
condacolab.install()

## Step 2: Install CadQuery, gmsh, and CalculiX via Conda

Run this cell **after** the kernel restarts from Step 1.

In [ ]:
# Verify condacolab is active, then install the heavy dependencies
import condacolab
condacolab.check()

# condacolab uses Python 3.11; install CadQuery for that env
!conda install -c conda-forge cadquery gmsh calculix -y


## Step 3: Clone the repository and install Python dependencies

In [ ]:
import os

REPO_URL = "https://github.com/DarthJarJarBinks-Meesa/auxetic_bone_implant.git"
REPO_DIR = "auxetic_bone_implant"

if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL}
else:
    print(f"'{REPO_DIR}' already exists — pulling latest changes.")

os.chdir(REPO_DIR)
!git pull origin main

# Install Python-only requirements
!pip install -r requirements.txt

print("\n✅ Repository ready.")

## Step 4: Configure PYTHONPATH

The pipeline uses absolute imports relative to `src/` (e.g., `from workflow.orchestrator import ...`). We need `src/` on the Python path.

In [ ]:
import sys
import os

# Add src/ to Python path for imports AND shell commands
src_path = os.path.join(os.getcwd(), "src")
if src_path not in sys.path:
    sys.path.insert(0, src_path)

os.environ["PYTHONPATH"] = "src"

print(f"✅ PYTHONPATH set to: {os.environ['PYTHONPATH']}")
print(f"✅ sys.path includes: {src_path}")

## Step 5: Verify installation

Quick smoke test to make sure all dependencies and imports work.

In [ ]:
# IMPORTANT: condacolab runs Python 3.11 in shell commands (!python)
# but the notebook kernel is Python 3.12.
# cadquery is installed for shell Python (3.11) — that's fine.
# Verify via shell to confirm:
!python -c "import cadquery as cq; print(f'✅ CadQuery {cq.__version__} (shell Python OK)')"


In [ ]:
# Verify all dependencies using conda's shell Python (not the notebook kernel)
# This is correct — the pipeline runs as shell commands too.
!python -c "
import cadquery as cq; print(f'CadQuery: {cq.__version__}')
import gmsh;           print(f'gmsh: {gmsh.__version__}')
import yaml;           print(f'PyYAML: {yaml.__version__}')
import numpy as np;    print(f'NumPy: {np.__version__}')
import matplotlib;     print(f'Matplotlib: {matplotlib.__version__}')
print('All dependencies OK')
"

# Verify internal project imports via the pipeline's plan command
!PYTHONPATH=src python -c "
from utils.config_loader import load_pipeline_config
cfg = load_pipeline_config()
print(f'Config loaded — project root: {cfg.project_root}')
print(f'Enabled designs:   {cfg.get_enabled_designs()}')
print(f'Enabled materials: {cfg.get_enabled_materials()}')
"

# Verify CalculiX
!which ccx && echo '✅ CalculiX found' || echo '⚠️  CalculiX not on PATH (export-only mode still works)'

print('\n🎉 All checks passed!')


## Step 6: Run the pipeline

Choose one of the options below:

| Mode | Cases | Description |
|------|-------|-------------|
| `--plan-only` | 0 (dry run) | Preview what cases will be generated |
| `--mode baseline_only` | ~12 | One case per design × material × load case |
| `--mode baseline_plus_one_factor_variation` | ~100-400 | Baseline + one-parameter-at-a-time variations |
| `--mode full_factorial` | ~1000+ | Full Cartesian product (may be slow!) |

In [ ]:
# Option A: Preview the case plan (no execution)
!PYTHONPATH=src python src/main.py --plan-only

In [ ]:
# Option B: Run baseline-only sweep (smallest, best for testing)
!PYTHONPATH=src python src/main.py --mode baseline_only --stop-on-first-failure

In [ ]:
# Option C: Run baseline + one-factor variation (default mode)
# !PYTHONPATH=src python src/main.py --mode baseline_plus_one_factor_variation

In [ ]:
# Option D: Run with solver execution enabled (requires CalculiX installed)
# !PYTHONPATH=src python src/main.py --mode baseline_only --run-solver --stop-on-first-failure

## Step 7: View results

After a successful run, results are in `runs/` and reports are in `reports/`.

In [ ]:
import os
from pathlib import Path

runs_dir = Path("runs")
reports_dir = Path("reports")

if runs_dir.exists():
    case_dirs = [d for d in runs_dir.iterdir() if d.is_dir()]
    print(f"📁 {len(case_dirs)} case(s) in runs/:")
    for d in sorted(case_dirs)[:20]:
        print(f"   {d.name}")
    if len(case_dirs) > 20:
        print(f"   ... and {len(case_dirs) - 20} more.")
else:
    print("⚠️  No runs/ directory found yet.")

print()

if reports_dir.exists():
    report_files = list(reports_dir.rglob("*"))
    print(f"📊 {len(report_files)} report file(s):")
    for f in sorted(report_files):
        if f.is_file():
            print(f"   {f.relative_to('.')}")
else:
    print("⚠️  No reports/ directory found yet.")